In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from functools import reduce

In [0]:
day1_path = "/Volumes/scd_inc_catalog/scd_schema/scd_vol/day1/customers_day1.csv" 

silver_path = "/Volumes/scd_inc_catalog/scd_schema/scd_vol_silver/"  #creating silver table 

In [0]:
#Creating a Reusable Function to Apply Transformations for the source file so every time no need to write transformations for each day

def clean_data(df):

    cleaned_df = (
        df
        .withColumnRenamed("email", "email_id")

        .withColumn("first_name", F.trim(F.col("first_name")))
        .withColumn("last_name", F.trim(F.col("last_name")))

        .withColumn(
            "category",
            F.trim(F.initcap(F.col("category")))
        )

        .withColumn(
            "city",
            F.trim(F.col("city"))
        )

        .withColumn(
            "payment_method",
            F.trim(F.col("payment_method"))
        )

        .withColumn(
            "product_id",
            F.trim(
                F.initcap(
                    F.col("product_id").cast("string")
                )
            )
        )

        .withColumn(
            "amount",
            F.col("amount").cast("double")
        )

        .withColumn(
            "quantity",
            F.col("quantity").cast("integer")
        )

        .withColumn(
            "customer_name",
            F.concat(
                F.col("first_name"),
                F.lit(" "),
                F.col("last_name")
            )
        )

        .drop("first_name", "last_name")

        .dropDuplicates(["order_id"])
    )

    return cleaned_df.select(
        "order_id",
        "customer_name",
        "product_id",
        "category",
        "quantity",
        "amount",
        "payment_method",
        "city",
        "email_id"
    )

In [0]:
day1_df = spark.read.csv(
    day1_path,
    header=True,
    inferSchema=True
) # Reading File from the Source_Path {day1_df we already mentioned the path in the above cell}

day1_clean = clean_data(day1_df) #using the function we created in the above cell to clean the data
display(day1_clean)

In [0]:
window_spec = Window.orderBy("order_id")

initial_silver = (
    day1_clean

    .withColumn(
        "order_sk",
        F.row_number().over(window_spec)
    )

    .withColumn(
        "start_date",
        F.to_date(F.lit("2026-04-01")) #creating start_date column with the previous date
    )

    .withColumn(
        "end_date",
        F.to_date(F.lit("9999-12-31")) #creating end_date column with the max date
    )

    .withColumn(
        "is_current",
        F.lit("Y")
    )
)

In [0]:
(
    initial_silver
    .write
    .format("delta")
    .mode("overwrite")
    .save(silver_path)
)
display(initial_silver) #Here all the transformations are done to the day1 file and saved a delta to the silver path with scd columns start_date,end_date,is_current

##The most important part — ONE SCD2 function

In [0]:

def process_incremental_file(file_path):

    print("=" * 60)
    print(f"Processing file: {file_path}")
    print("=" * 60)

    # --------------------------------------------------
    # 1. READ AND CLEAN INCREMENTAL FILE
    # --------------------------------------------------

    source_df = spark.read.csv(
        file_path,
        header=True,
        inferSchema=True
    )

    incremental_clean = (
        source_df
        .withColumnRenamed("email", "email_id")
        .withColumn("first_name", F.trim(F.col("first_name")))
        .withColumn("last_name", F.trim(F.col("last_name")))
        .withColumn("category", F.trim(F.initcap(F.col("category"))))
        .withColumn("city", F.trim(F.col("city")))
        .withColumn("payment_method", F.trim(F.col("payment_method")))
        .withColumn(
            "product_id",
            F.trim(F.initcap(F.col("product_id").cast("string")))
        )
        .withColumn("amount", F.col("amount").cast("double"))
        .withColumn("quantity", F.col("quantity").cast("integer"))
        .withColumn(
            "customer_name",
            F.concat(
                F.col("first_name"),
                F.lit(" "),
                F.col("last_name")
            )
        )
        .drop("first_name", "last_name")
        .dropDuplicates(["order_id"])
        .select(
            "order_id",
            "customer_name",
            "email_id",
            "city",
            "product_id",
            "category",
            "amount",
            "quantity",
            "payment_method"
        )
    )

    print("Incoming records:", incremental_clean.count())


    # --------------------------------------------------
    # 2. READ EXISTING SILVER DELTA TABLE
    # --------------------------------------------------

    silver_df = (
        spark.read
        .format("delta")
        .load(silver_path)
    )

    current_silver = silver_df.filter(
        F.col("is_current") == "Y"
    )

    print("Existing current records:", current_silver.count())


    # --------------------------------------------------
    # 3. IDENTIFY NEW RECORDS
    # --------------------------------------------------

    new_records = (
        incremental_clean.alias("src")
        .join(
            current_silver.alias("tgt"),
            F.col("src.order_id") == F.col("tgt.order_id"),
            "left"
        )
        .filter(F.col("tgt.order_id").isNull())
        .select("src.*")
    )

    print("New records:", new_records.count())


    # --------------------------------------------------
    # 4. IDENTIFY CHANGED RECORDS
    # --------------------------------------------------

    compare_columns = [
        "customer_name",
        "email_id",
        "city",
        "product_id",
        "category",
        "amount",
        "quantity",
        "payment_method"
    ]

    change_condition = reduce(
        lambda a, b: a | b,
        [
            ~F.col(f"src.{c}").eqNullSafe(F.col(f"tgt.{c}"))
            for c in compare_columns
        ]
    )

    changed_records = (
        incremental_clean.alias("src")
        .join(
            current_silver.alias("tgt"),
            F.col("src.order_id") == F.col("tgt.order_id"),
            "inner"
        )
        .filter(change_condition)
        .select("src.*")
    )

    print("Changed records:", changed_records.count())


    # --------------------------------------------------
    # 5. IDENTIFY UNCHANGED RECORDS
    # --------------------------------------------------

    unchanged_records = (
        incremental_clean.alias("src")
        .join(
            current_silver.alias("tgt"),
            F.col("src.order_id") == F.col("tgt.order_id"),
            "inner"
        )
        .filter(~change_condition)
        .select("src.*")
    )

    print("Unchanged records:", unchanged_records.count())


    # --------------------------------------------------
    # 6. EXPIRE OLD VERSIONS
    # --------------------------------------------------

    changed_ids = (
        changed_records
        .select("order_id")
        .distinct()
    )

    expired_silver = (
        silver_df.alias("tgt")
        .join(
            changed_ids.alias("chg"),
            F.col("tgt.order_id") == F.col("chg.order_id"),
            "left"
        )
        .withColumn(
            "end_date",
            F.when(
                (F.col("chg.order_id").isNotNull()) &
                (F.col("tgt.is_current") == "Y"),
                F.date_sub(F.current_date(), 1)
            ).otherwise(F.col("tgt.end_date"))
        )
        .withColumn(
            "is_current",
            F.when(
                (F.col("chg.order_id").isNotNull()) &
                (F.col("tgt.is_current") == "Y"),
                F.lit("N")
            ).otherwise(F.col("tgt.is_current"))
        )
        .select(
            "tgt.order_sk",
            "tgt.order_id",
            "tgt.customer_name",
            "tgt.product_id",
            "tgt.category",
            "tgt.quantity",
            "tgt.amount",
            "tgt.payment_method",
            "tgt.city",
            "tgt.email_id",
            "tgt.start_date",
            "end_date",
            "is_current"
        )
    )


    # --------------------------------------------------
    # 7. COMBINE NEW AND CHANGED RECORDS
    # --------------------------------------------------

    records_to_insert = (
        new_records
        .unionByName(changed_records)
    )

    print("Total records to insert:", records_to_insert.count())


    # --------------------------------------------------
    # 8. GENERATE NEW SURROGATE KEYS
    # --------------------------------------------------

    max_sk = (
        silver_df
        .agg(F.max("order_sk").alias("max_sk"))
        .first()["max_sk"]
    )

    if max_sk is None:
        max_sk = 0

    new_versions_window = Window.orderBy("order_id")

    new_versions = (
        records_to_insert
        .withColumn(
            "order_sk",
            F.row_number().over(new_versions_window) + max_sk
        )
        .withColumn(
            "start_date",
            F.current_date()
        )
        .withColumn(
            "end_date",
            F.to_date(F.lit("9999-12-31"))
        )
        .withColumn(
            "is_current",
            F.lit("Y")
        )
        .select(
            "order_sk",
            "order_id",
            "customer_name",
            "product_id",
            "category",
            "quantity",
            "amount",
            "payment_method",
            "city",
            "email_id",
            "start_date",
            "end_date",
            "is_current"
        )
    )


    # --------------------------------------------------
    # 9. CREATE FINAL SILVER DATAFRAME
    # --------------------------------------------------

    final_silver_df = (
        expired_silver
        .unionByName(new_versions)
    )

    print(
        "Final total records:",
        final_silver_df.count()
    )

    print(
        "Final current records:",
        final_silver_df
        .filter(F.col("is_current") == "Y")
        .count()
    )


    # --------------------------------------------------
    # 10. SAVE BACK TO DELTA TABLE
    # --------------------------------------------------

    (
        final_silver_df
        .write
        .format("delta")
        .mode("overwrite")
        .save(silver_path)
    )

    print("Successfully processed:", file_path)
    print("Silver Delta table updated!")

    return final_silver_df

In [0]:
# def process_incremental_file(file_path):

#     print("=" * 80)
#     print(f"Processing file: {file_path}")
#     print("=" * 80)

#     # ------------------------------------------------
#     # 1. Read incremental file
#     # ------------------------------------------------

#     source_df = (
#         spark.read
#         .csv(
#             file_path,
#             header=True,
#             inferSchema=True
#         )
#     )

#     print("Source records:", source_df.count())


#     # ------------------------------------------------
#     # 2. Clean source
#     # ------------------------------------------------

#     incremental_clean = clean_data(source_df)

#     print(
#         "Clean incremental records:",
#         incremental_clean.count()
#     )             #we using clean_data function to clean the data 


#     # ------------------------------------------------
#     # 3. Read existing Silver
#     # ------------------------------------------------

#     silver_df = (
#         spark.read
#         .format("delta")
#         .load(silver_path)
#     )        


#     # ------------------------------------------------
#     # 4. Get current records
#     # ------------------------------------------------

#     current_silver = (
#         silver_df
#         .filter(F.col("is_current") == "Y")
#     )

#     print(
#         "Current Silver records:",
#         current_silver.count()
#     )


#     # ------------------------------------------------
#     # 5. Find NEW records
#     # ------------------------------------------------

#     new_records = (
#         incremental_clean.alias("src")

#         .join(
#             current_silver.alias("tgt"),

#             F.col("src.order_id") ==
#             F.col("tgt.order_id"),

#             "left"
#         )

#         .filter(
#             F.col("tgt.order_id").isNull()
#         )

#         .select("src.*")
#     )

#     print(
#         "New records:",
#         new_records.count()
#     )


#     # ------------------------------------------------
#     # 6. Find CHANGED records
#     # ------------------------------------------------

#     changed_records = (
#         incremental_clean.alias("src")

#         .join(
#             current_silver.alias("tgt"),

#             F.col("src.order_id") ==
#             F.col("tgt.order_id"),

#             "inner"
#         )

#         .filter(

#             (
#                 F.coalesce(
#                     F.col("src.customer_name"),
#                     F.lit("")
#                 )
#                 !=
#                 F.coalesce(
#                     F.col("tgt.customer_name"),
#                     F.lit("")
#                 )
#             )

#             |

#             (
#                 F.coalesce(
#                     F.col("src.product_id"),
#                     F.lit("")
#                 )
#                 !=
#                 F.coalesce(
#                     F.col("tgt.product_id"),
#                     F.lit("")
#                 )
#             )

#             |

#             (
#                 F.coalesce(
#                     F.col("src.category"),
#                     F.lit("")
#                 )
#                 !=
#                 F.coalesce(
#                     F.col("tgt.category"),
#                     F.lit("")
#                 )
#             )

#             |

#             (
#                 F.coalesce(
#                     F.col("src.quantity"),
#                     F.lit(-1)
#                 )
#                 !=
#                 F.coalesce(
#                     F.col("tgt.quantity"),
#                     F.lit(-1)
#                 )
#             )

#             |

#             (
#                 F.coalesce(
#                     F.col("src.amount"),
#                     F.lit(-1.0)
#                 )
#                 !=
#                 F.coalesce(
#                     F.col("tgt.amount"),
#                     F.lit(-1.0)
#                 )
#             )

#             |

#             (
#                 F.coalesce(
#                     F.col("src.payment_method"),
#                     F.lit("")
#                 )
#                 !=
#                 F.coalesce(
#                     F.col("tgt.payment_method"),
#                     F.lit("")
#                 )
#             )

#             |

#             (
#                 F.coalesce(
#                     F.col("src.city"),
#                     F.lit("")
#                 )
#                 !=
#                 F.coalesce(
#                     F.col("tgt.city"),
#                     F.lit("")
#                 )
#             )

#             |

#             (
#                 F.coalesce(
#                     F.col("src.email_id"),
#                     F.lit("")
#                 )
#                 !=
#                 F.coalesce(
#                     F.col("tgt.email_id"),
#                     F.lit("")
#                 )
#             )
#         )

#         .select("src.*")
#     )

#     print(
#         "Changed records:",
#         changed_records.count()
#     )


#     # ------------------------------------------------
#     # 7. Get changed IDs
#     # ------------------------------------------------

#     changed_ids = (
#         changed_records
#         .select("order_id")
#         .distinct()
#     )


#     # ------------------------------------------------
#     # 8. Expire old versions
#     # ------------------------------------------------

#     expired_silver = (
#         silver_df.alias("tgt")

#         .join(
#             changed_ids.alias("chg"),

#             F.col("tgt.order_id") ==
#             F.col("chg.order_id"),

#             "left"
#         )

#         .filter(
#             F.col("tgt.is_current") == "Y"
#         )

#         .withColumn(
#             "end_date",

#             F.when(
#                 F.col("chg.order_id").isNotNull(),

#                 F.date_sub(
#                     F.current_date(),
#                     1
#                 )
#             )

#             .otherwise(
#                 F.col("tgt.end_date")
#             )
#         )

#         .withColumn(
#             "is_current",

#             F.when(
#                 F.col("chg.order_id").isNotNull(),

#                 F.lit("N")
#             )

#             .otherwise(
#                 F.col("tgt.is_current")
#             )
#         )

#         .select(
#             "tgt.order_sk",
#             "tgt.order_id",
#             "tgt.customer_name",
#             "tgt.product_id",
#             "tgt.category",
#             "tgt.quantity",
#             "tgt.amount",
#             "tgt.payment_method",
#             "tgt.city",
#             "tgt.email_id",
#             "tgt.start_date",
#             "end_date",
#             "is_current"
#         )
#     )


#     # ------------------------------------------------
#     # 9. Combine NEW + CHANGED
#     # ------------------------------------------------

#     records_to_insert = (
#         new_records
#         .unionByName(changed_records)
#     )


#     # ------------------------------------------------
#     # 10. Generate surrogate keys
#     # ------------------------------------------------

#     max_sk = (
#         silver_df
#         .agg(
#             F.max("order_sk").alias("max_sk")
#         )
#         .first()["max_sk"]
#     )

#     if max_sk is None:
#         max_sk = 0


#     new_records_window = Window.orderBy("order_id")


#     # ------------------------------------------------
#     # 11. Create new SCD2 versions
#     # ------------------------------------------------

#     new_versions = (

#         records_to_insert

#         .withColumn(
#             "order_sk",

#             F.row_number()
#             .over(new_records_window)
#             + max_sk
#         )

#         .withColumn(
#             "start_date",
#             F.current_date()
#         )

#         .withColumn(
#             "end_date",
#             F.to_date(
#                 F.lit("9999-12-31")
#             )
#         )

#         .withColumn(
#             "is_current",
#             F.lit("Y")
#         )

#         .select(
#             "order_sk",
#             "order_id",
#             "customer_name",
#             "product_id",
#             "category",
#             "quantity",
#             "amount",
#             "payment_method",
#             "city",
#             "email_id",
#             "start_date",
#             "end_date",
#             "is_current"
#         )
#     )


#     # ------------------------------------------------
#     # 12. Final Silver
#     # ------------------------------------------------

#     final_silver_df = (
#         expired_silver
#         .unionByName(new_versions)
#     )


#     # ------------------------------------------------
#     # 13. Write back to same Silver
#     # ------------------------------------------------

#     (
#         final_silver_df

#         .write
#         .format("delta")

#         .mode("overwrite")

#         .option(
#             "overwriteSchema",
#             "true"
#         )

#         .save(silver_path)
#     )


#     # ------------------------------------------------
#     # 14. Validation
#     # ------------------------------------------------

#     final_df = (
#         spark.read
#         .format("delta")
#         .load(silver_path)
#     )

#     print(
#         "Final Silver records:",
#         final_df.count()
#     )

#     print(
#         "Current records:",
#         final_df
#         .filter(F.col("is_current") == "Y")
#         .count()
#     )

#     print(
#         "Historical records:",
#         final_df
#         .filter(F.col("is_current") == "N")
#         .count()
#     )

#     return final_df

In [0]:
day2_path = "/Volumes/scd_inc_catalog/scd_schema/scd_vol/day2/customers_day2.csv"

process_incremental_file(day2_path)

In [0]:
display(
    spark.read
    .format("delta")
    .load(silver_path))

In [0]:
day3_path = "/Volumes/scd_inc_catalog/scd_schema/scd_vol/day3/customers_day3.csv"

process_incremental_file(day3_path)

In [0]:
display(
    spark.read
    .format("delta")
    .load(silver_path))